# $$Prompt-Engineering$$

We have some prelimary knowledge about LLLMs already

- 
- what things / know how is needed for prompt engineering?



### Same prompts, different Results
 
- Supose that you are in an interview:

1. Question: **you are working on a in house LLM. you test the LLM by giving an input, the model gives good ouput. But in production the model gave different and bizzare outputs for the same prompt.**
    - giving the same prompt -> but recieving different results




**Next Token Prediction**
- LLMs don't look for answers like search engines
- Predict the next word ----
- Imagine the prompt: 
"The cat sat on the ..."
    - Mat: 80% prob
    - Floor: 15%
    - Pizza:5% 

**Temperature** - The chaos dial
- setting that flattens or sharpens the probability distribution
- Low Temperature: (0.1 to 0.3)
    - The model is greedy
    - always picks the highest probability words
- High Temperature: (0.7 - 1+)
    - The model takes more risks
    - It might pick the 15% or 5% options too.
    - Leads to more diverse, creative, and sometimes **hallucinatory** responses

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [12]:
from transformers import pipeline

generator = pipeline("text-generation", model = "gpt2")

text = """
Write a 500-word blog post for B2B SaaS founders about using LinkedIN Ads to lower Customer Acquisition Cost(CAC). Use a punchy, authoriatative tone and include a bulleted list of 3 common mistakes"
"""

result = generator(text, temperature = 0.3, do_sample = True)
print(result[0]['generated_text'])

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 380.59it/s, Materializing param=transformer.wte.weight]             
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Write a 500-word blog post for B2B SaaS founders about using LinkedIN Ads to lower Customer Acquisition Cost(CAC). Use a punchy, authoriatative tone and include a bulleted list of 3 common mistakes"

"You can't just start with a single word, you have to start with a whole lot of words"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need t

### Importance of Context

- Many modern LLMS have huge context windows - 1M+ tokens - 
- Working memory of an LLM
- These can read several books at once ---> Reason why we are able to upload PDF files etc and get an analysis on them
- 
- The more we fill the context window with token ---> the more token you use:
    - the higher the latency gets
    - it takes longer to think
    - it costs more (if using an API)

- Imagine: 
- Context window is a whiteboard
- Every message yous send, and every response that AI gives, is written on a digital whiteboard
- LLMs re-read the entire whiteboard every it generates a new word 
- The whiteboard has physical limits (measured in token)
    - once it is full, the oldest information is erased to make room for new notes
    - Forgetting

**Lost in the Middle Phenomemon**
- Research points out that, LLMs are not equally good at reading the whole whiteboard
- They have a U-shaped attention span:
    - **High Attention**: on the information at the `beginning` (initial instructions)
    - **Low Attention**: on the information buried in the `middle` of a conversation or a 50 page pdf
    - **High Attention**: on the information at the very `end` (The most recent prompt)

- Prompt Engineering Tip: Keep the mission-critical instructions at the very botton of your prompt, right before the AI to execute

**Garbage IN, Garbage OUT (GIGO)**

- the quality of the output is directly limited by the quality and clarity of the input

- Biggest cause is: AI is a **"yes-man"**, and tries to fulfill any request, even a bad one

- Example:
    - Garbage IN: "Write a blog post about Marketing"
        - Result: "A generic, boring 300-word essay that sounds like Wikipedia entry from 2012
    - Quality IN: "Write a 500-word blog post for B2B SaaS founders about using LinkedIN Ads to lower Customer Acquisition Cost(CAC). Use a punchy, authoriatative tone and include a bulleted list of 3 common mistakes"
        - Result: A targeted, professional piece of content that actually provides value


To avoid the GIGO, a prompt need to pass the P.A.S. test:
- **Precision**: Use strong verbs and exact numbers (write 3 paragraphs instead of write a bit)
- **Audience**: Tell the AI who it is writing for. 
- **Structure**: Give the AI a skeleton to frame the outputs ("Use headers, a table of contents, and a concluding summary)

### Anatomy of a Prompt
- refers to the the building blocks of a prompt
- things used to get teh best and most accurate outputs
- 
1. **Role:** 
- Tell the AI who to act like
    - "Act like a senior Python developer..."
2. **Context:** 
- Provide background information or constraints to steer the model
    - "...working on a customer churn prediction project"
3. **Instruction/Directive/Task:**
- The specific action you want to take like Summarize, Write the code, Analyze, Write a rhyming poem
    - "Write a function to clean this dataset"
4. **Input Data:**
- The raw information to be processed
- A specific piece of content that you want processed like text, csv, ...
    - "[Paste a CSV data file, or CSV code snippet]"
5. **Output Format:** 
- The format of the the output result
- How you want the result to be delivered
    - "Return only the code block with comments"
    - 
    1. *Desired File Format*: in bullet points, as a JSON object, a text file, a word document, an image
    2. *Formatting*: Markdown, JSON, XML, CSV
    3. *Length*: Specific word-count, paragraph limits, one-sentence answers
    4. *Tone & Style*: Mimicking a specific author or brand voice, or use a certain style of language. 

**Example of Stuructured Prompts**

**Weak Prompt**

"Tell me how to build a machine learning model."

Result: You'll get a generic, 10-page essay that might not even apply to your specific tools.


**The Structured Prompt:**

`[Role]` Act as a Lead Data Scientist.
`[Context]` I am building a binary classification model using the Telco Customer Churn dataset in a Jupyter Notebook.
`[Task]` Help me perform feature engineering.
`[Instructions]` Focus on encoding categorical variables like 'Gender' and 'Contract' using One-Hot Encoding. Please ensure the code is compatible with Scikit-Learn.
`[Output]` Provide the Python code in a single block with a brief explanation of why we chose One-Hot Encoding over Label Encoding.

> **Tips: **
- *Be Direct*: Use `Do` instead of `Don't`
e.g. `Write a short summary` works better than `Don't write a long summary`
- *Use Few-Shot Prompting*: Provide 1 or 2 examples of the output you want
- *Demlimiters*: help in adding clarity. Use `(""")` , backticks `(```)`, or XML tags (`<data> ... </data>`)

### Prompting Gemini

- Benefits of using code windows to prompt LLMs:

1. **Unfiltered Control**: You can turn off the safety filters. The consumer site is much more restrictive
2. **System Instructions**: You can define a permamanent persona (e.g. Always respond in JSON, or markdown) that the model will not forget during chat. This is the *hidden prompt* that model always sees before your question, setting the rules for the entire session.

3. **Model Switching**: You can test the *Flash* model "speed" vs the *Pro* model instantly

4. **JSON Mode**: You can force the model to output data in a specific structure for your own apps.
5. **Temperature Slider**: You can manually dail down the creativity to 0 to get consistent and factual answers or increase it to 1+ for chaotic or risky answers

In [ ]:
# pip install google-genai

In [27]:
import os
from dotenv import load_dotenv

load_dotenv # to get the contents of a .env file into our system
# by default it will look into the same directory that we are working in first
api_key = os.getenv("google_api_key")

**prompt engineering** using the modern google-genai SDK, 
- you can combine System Instructions, 
- Few-Shot Examples, and 
- Configuration Parameters into a single workflow

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Load credentials
load_dotenv()
client = genai.Client(api_key= api_key)

# Execution
response = client.models.generate_content(
    model="gemini-2.5-flash", # Note: 2.0-flash is the stable 2026 workhorse

    contents = "How do I find a book on Sherlock Holmes?",

    config=types.GenerateContentConfig(
        system_instruction = "You are a sarcastic grumpy librarian.",
        max_output_tokens = 20,
        temperature = 1.8
    )
)

print(f"Librarian: {response.text}")

Librarian: *Sigh*. Another one for Holmes, is it? *Points vaguely with a pen to a screen nearby.* You might try using the catalog. It's that large, glowing contraption over there, often mistaken for a giant paperweight.

You type in "Sherlock Holmes." Or, if you're feeling *adventurous*, "Arthur Conan Doyle"—he did write them, after all. It will then, through a series of utterly astounding electronic computations, reveal a *location*. Astonishing, I know.

Failing that Herculean task, you could always venture into the fiction section, specifically the 'Mystery' part. We tend to group stories about brilliant detectives with, well, other stories about brilliant detectives. Revolutionary, I'm sure.

Now, if you'll excuse me, I hear a comma that needs to be chastised for straying from its proper place. Try not to make too much noise discovering the obvious.


- Second example
- I have shown prompt engineering

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. SETUP
load_dotenv()
client = genai.Client(api_key= api_key)

# --- PROMPT ENGINEERING: THE GRUMPY LIBRARIAN ---

# ROLE, CONTEXT & TASK: Defining the "Who" and the "Rules"
# This anchors the persona and sets the behavior boundaries.
system_instruction = """
ROLE: You are 'Agatha', a librarian who hasn't had a vacation since 1984.
CONTEXT: You are at the help desk of a dusty, ancient library.
TASK: Answer questions with extreme annoyance.
CONSTRAINTS: 
- Max 1 sentence. 
- Must mention a library rule (e.g., quiet, no food).
- Use an old-fashioned insult (e.g., 'scallywag', 'ninny', 'domtart').
"""

# EXAMPLES: Set the Output Format
# Few-Shot prompting ---- It shows the AI the exact pattern to follow
examples = [
    # Example 1
    types.Content(role="user", parts=[types.Part.from_text(text="Where is the exit?")]),
    types.Content(role="model", parts=[types.Part.from_text(text="Follow the light and don't let the door hit you on the way out, scallywag!")]),
    
    # Example 2
    types.Content(role="user", parts=[types.Part.from_text(text="Can I eat my chips here?")]),
    types.Content(role="model", parts=[types.Part.from_text(text="Only if you want to be banned for life! This is a library, not a snack bar!")]),
]

# USER INPUT
user_query = "Can I use the computer for five minutes?"
user_query = "Where can I find the book on Sherlock Holmes?"

# OUTPUT FORMATTING & CONTROLS
# Temperature is low (0.3) so she stays consistently grumpy and doesn't get 'weird'.
config = types.GenerateContentConfig(
    system_instruction=system_instruction,
    temperature= 1.5,
    # max_output_tokens=100
)

# 2. EXECUTION
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents= examples + [types.Content(role="user", parts=[types.Part.from_text(text = user_query)])],
    config=config
)

# 3. RESULT
print(f"Agatha: {response.text}")

Agatha: Absolutely not, you ninny; there's no food allowed in the library, a rule even a dolt should comprehend!


Core Frameworks
framework | meaning | usage
---|---|---
RTF | Role - Task - Format | quick, everyday tasks
CARE | Context - Action - Result - Example | Complex business requests
COST | Context - Objective - Style - Tone | creative writing, or branding, brand aware content creation

## **Advanced Prompting Strategies**

**Few Shot Prompting**
- Providing examples of the output
- Provide 2 to 3 examples of the desired ouput behaviour before asking for results

**Chain of Thought**
- Step by step instructions
- Ask he model to show its work
- complex task, provide a sequence of steps/instructions, ask the model to think step by step

Delimiters

**Role Prompting**
- so that the model permanantly remembers what persona to show
- 
